# Extracting new-product announcements from SEC Form 8-K filings with a local LLM

**Repository:** https://github.com/alexhubenthal/sec-8k-product-extraction

This notebook walks through the pipeline implemented in the `sec8k` package step by step on one
company, then loads the results of the full run (`output/new_products.csv`).

Pipeline: S&P 500 tickers → CIK (`company_tickers.json`) → 8-K list (browse-edgar Atom feed) →
filing documents → plain text → **LLM named-entity extraction** (Ollama + Llama 3, JSON-schema
constrained) → validation and LLM verification → CSV.

Requirements: the Ollama server running with the `llama3` model, and the environment variable
`SEC_USER_AGENT="Your Name your.email@example.com"` (SEC requires it).

In [ ]:
import json
import logging
import os
import sys
from pathlib import Path

import pandas as pd

# run from the repository root so that data/ and output/ resolve
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

from sec8k import edgar, llm, pipeline

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
pd.set_option("display.max_colwidth", 120)

USER_AGENT = os.environ.get("SEC_USER_AGENT", "")
assert "@" in USER_AGENT, "set SEC_USER_AGENT='Your Name your.email@example.com' before running"
client = edgar.EdgarClient(USER_AGENT, cache_dir="data/raw/http_cache")
ollama = llm.OllamaClient(model="llama3", num_ctx=8192)
ollama.check_available()
print("SEC client and Ollama ready")

## Step 1 – Ticker → CIK

`https://www.sec.gov/files/company_tickers.json` lists every registrant with a ticker. The CIK is
zero-padded to ten digits, the form EDGAR URLs expect.

In [ ]:
ticker_map = edgar.load_ticker_map(client)
print(len(ticker_map), "tickers in company_tickers.json")
for symbol in ["AAPL", "LLY", "BRK.B"]:
    print(symbol, "->", edgar.lookup_cik(ticker_map, symbol))

## Step 2 – Recent 8-K filings of a company (Atom feed)

`https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK={cik}&type=8-K&count={count}&output=atom`
returns the filings with their timestamp, accession number and the 8-K *items* reported. Items 2.02
(results of operations), 7.01 (Regulation FD) and 8.01 (other events) are the ones that carry press
releases, so those are the filings worth reading.

In [ ]:
TICKER = "LLY"
record = edgar.lookup_cik(ticker_map, TICKER)
filings = edgar.list_8k_filings(client, record["cik"], record["ticker"], record["title"], count=10)
pd.DataFrame([f.to_dict() for f in filings])[["accession", "filing_type", "filing_date", "filed_at", "items"]]

## Step 3 – Filing documents → plain text

The filing index page lists the documents; the primary 8-K (served through the inline-XBRL viewer,
so the `/ix?doc=` prefix is removed) and the EX-99 press-release exhibits are downloaded and converted
to text with BeautifulSoup. The text handed to the model is capped at 10,000 characters.

In [ ]:
candidate = next(f for f in filings if not f.is_amendment and f.has_any_item(edgar.PRESS_RELEASE_ITEMS))
filing_text = edgar.download_filing_text(client, candidate, max_chars=10_000)
print("documents:", [(d.doc_type, d.filename) for d in filing_text.documents if d.is_primary or d.is_press_release])
print("accepted at:", filing_text.accepted_at, "| text chars:", len(filing_text.text), "| cue score:", filing_text.cue_score)
print()
print(filing_text.text[:1500], "...")

## Step 4 – LLM named-entity extraction

The model is instructed as an NER system with two entity types, **ORGANIZATION** (the registrant)
and **PRODUCT** (a product the registrant announces as new). Ollama's structured output constrains
the answer to a JSON schema in which the verbatim `evidence` sentence comes *before* the product name,
so the entity is extracted from a quote. `postprocess()` then grounds every entity in the text
(recording its character span), checks the evidence and drops hallucinations.

In [ ]:
raw, response = llm.extract_entities(
    ollama, filing_text.text, ticker=candidate.ticker, edgar_name=candidate.company,
    filing_date=candidate.filing_date, items=candidate.items,
)
print(f"LLM answered in {response.seconds:.1f}s ({response.prompt_tokens} prompt tokens, {response.completion_tokens} output tokens)")
print(json.dumps(raw, indent=2)[:2500])

In [ ]:
cleaned = llm.postprocess(raw, filing_text.text, ticker=candidate.ticker,
                          fallback_company=record["title"], edgar_title=record["title"])
print("ORGANIZATION:", cleaned.company_name)
for product in cleaned.products:
    start, end = product["span"] or (0, 0)
    print(f"PRODUCT: {product['new_product']!r} [{product['announcement_type']}] span={product['span']}")
    print("   description:", product["product_description"])
    print("   in context  : ...", filing_text.text[max(0, start - 60):end + 60].replace("\n", " "), "...")
for product in cleaned.rejected:
    print("REJECTED:", product["new_product"], "->", product["reason"])

## Step 5 – Second-pass verification

Each surviving product is re-checked by a short LLM call that only sees the passage around the
mention and must confirm that the filer is announcing it as new (not an existing product, a drug in
trials, a partner's product, ...).

In [ ]:
for product in cleaned.products:
    context = llm.context_window(filing_text.text, product["span"])
    verdict, reason, seconds = llm.verify_product(ollama, cleaned.company_name, product["new_product"],
                                                  product["product_description"], context)
    print(f"{product['new_product']!r}: verified={verdict} ({seconds:.1f}s) - {reason}")

## Step 6 – The full run and its results

The whole pipeline (all S&P 500 companies, checkpointing, CSV outputs) is run from the command line:

```bash
python run_extraction.py            # stops after 100 product rows
```

or programmatically with `pipeline.run(pipeline.PipelineConfig(user_agent=USER_AGENT))`.
Below, the outputs of the submitted run are loaded.

In [ ]:
products = pd.read_csv("output/new_products.csv")
print(products.shape[0], "product rows;", products.stock_name.nunique(), "companies; longest description:",
      products.product_description.str.len().max(), "chars")
products.head(25)

In [ ]:
filings_log = pd.read_csv("output/filings_processed.csv")
summary = json.load(open("output/summary.json"))
print(json.dumps({k: summary[k] for k in ["model", "filings_processed", "filings_with_new_product", "product_rows",
                                           "companies_processed", "companies_with_product", "rejected_products",
                                           "llm_seconds_mean", "verify_seconds_mean_per_filing", "llm_seconds_total",
                                           "filing_date_min", "filing_date_max"]}, indent=2))
print("\nfilings with a product by 8-K item:", summary["items_of_filings_with_product"])
print("announcement types:", summary["announcement_types"])
print("rejection reasons:", summary["rejection_reasons"])

In [ ]:
(filings_log.assign(product=filings_log.n_products > 0)
            .groupby("sector")["product"].agg(filings="size", with_product="sum")
            .sort_values("with_product", ascending=False))

In [ ]:
# the same table in the pipe-separated layout requested in the assignment
print(open("output/new_products_table.txt").read()[:3000])